# Python 面向对象编程完全指南

**面向对象**是一种以对象为中心组织代码的思想。对象拥有**属性**和**行为**，而**类**是创建对象的“模板”。

本教程将系统讲解：
- 类与实例
- 实例属性、类属性
- 实例方法、类方法、静态方法
- 继承与多重继承
- 权限控制（公有、受保护、私有）
- getter / setter
- 魔法方法
- 多态（标准多态与鸭子类型）
- 抽象类
- 综合案例：学生成绩管理系统

## 1. 类的定义与创建实例

使用 `class` 关键字定义类，类名通常采用**大驼峰命名法**。
`__init__` 是初始化方法，在创建实例时自动调用，用于绑定属性。

In [1]:
# 定义 Person 类
class Person:
    def __init__(self, name, age, gender):
        self.name = name          # 公有属性
        self.age = age
        self.gender = gender

# 创建实例（实例化）
p1 = Person('张三', 18, '男')
p2 = Person('李四', 22, '女')

# 通过点语法访问属性
print(p1.name, p1.age, p1.gender)
print(p2.name, p2.age, p2.gender)

# 查看实例的所有属性
print(p1.__dict__)
print(p2.__dict__)

# 查看实例的类型
print(type(p1))   # <class '__main__.Person'>

张三 18 男
李四 22 女
{'name': '张三', 'age': 18, 'gender': '男'}
{'name': '李四', 'age': 22, 'gender': '女'}
<class '__main__.Person'>


## 2. 自定义方法（行为）

类中定义的函数称为**方法**。实例方法的第一个参数是 `self`，代表调用该方法的实例本身。

In [2]:
class Person:
    def __init__(self, name, age, gender):
        self.name = name
        self.age = age
        self.gender = gender

    def speak(self, msg):
        print(f'我叫{self.name}，年龄{self.age}，性别{self.gender}，我想说：{msg}')

    def run(self, distance):
        print(f'{self.name} 跑了 {distance} 米')


p = Person('王五', 25, '男')
p.speak('你好')
p.run(500)

# 方法查找顺序：实例自身 → 类 → 父类
# 类身上存储了方法定义，实例身上只有属性

我叫王五，年龄25，性别男，我想说：你好
王五 跑了 500 米


## 3. 实例属性 vs 类属性

- **实例属性**：通过 `self.xxx = value` 定义在实例身上，各个实例互不影响。
- **类属性**：直接在类体中赋值，被所有实例共享，通常用于存放公共常量。

In [3]:
# 实例属性：每个实例独立
class Demo:
    def __init__(self, x):
        self.x = x   # 实例属性

d1 = Demo(10)
d2 = Demo(20)
print(d1.x, d2.x)   # 10 20

# 类属性：所有实例共享
class Person:
    max_age = 120          # 类属性
    planet = '地球'

    def __init__(self, name, age):
        self.name = name
        if age > Person.max_age:
            self.age = Person.max_age
        else:
            self.age = age

p1 = Person('张三', 150)
p2 = Person('李四', 30)
print(p1.age)          # 120
print(p2.age)          # 30
print(Person.max_age)  # 通过类访问类属性
print(p1.planet)       # 通过实例访问类属性

# 注意：通过 【实例.类属性名 = 值】 会创建实例属性，而非修改类属性
p1.planet = '火星'
print(p1.planet)       # 火星（实例属性）
print(p2.planet)       # 地球（仍为类属性）
print(Person.planet)   # 地球

10 20
120
30
120
地球
火星
地球
地球


## 4. 实例方法、类方法、静态方法

- **实例方法**：第一个参数为 `self`，通常由实例调用，可访问实例属性和类属性。
- **类方法**：用 `@classmethod` 修饰，第一个参数为 `cls`（类本身），推荐由类调用。
- **静态方法**：用 `@staticmethod` 修饰，没有 `self` 或 `cls`，与普通函数类似，但放在类中便于组织。

In [4]:
from datetime import datetime

class Person:
    planet = '地球'

    def __init__(self, name, age, gender):
        self.name = name
        self.age = age
        self.gender = gender

    # 实例方法
    def speak(self, msg):
        print(f'{self.name}: {msg}')

    # 类方法
    @classmethod
    def change_planet(cls, new_planet):
        cls.planet = new_planet

    @classmethod
    def create_from_string(cls, info_str):
        """工厂方法：从 '姓名-出生年-性别' 创建实例"""
        name, birth_year, gender = info_str.split('-')
        age = datetime.now().year - int(birth_year)
        return cls(name, age, gender)

    # 静态方法
    @staticmethod
    def is_adult(birth_year):
        return datetime.now().year - birth_year >= 18

    @staticmethod
    def mask_idcard(idcard):
        return idcard[:6] + '****' + idcard[-4:]


# 类方法调用
Person.change_planet('火星')
print(Person.planet)   # 火星

p = Person.create_from_string('王五-2000-男')
print(p.__dict__)

# 静态方法调用
print(Person.is_adult(2010))   # False
print(Person.mask_idcard('110101199001011234'))

火星
{'name': '王五', 'age': 26, 'gender': '男'}
False
110101****1234


## 5. 继承

子类可以继承父类的属性和方法，实现代码复用。使用 `super()` 调用父类方法。

In [5]:
# 基本继承
class Person:
    def __init__(self, name, age, gender):
        self.name = name
        self.age = age
        self.gender = gender

    def speak(self):
        print(f'我是{self.name}')

class Student(Person):
    def __init__(self, name, age, gender, stu_id, grade):
        super().__init__(name, age, gender)  # 调用父类初始化
        self.stu_id = stu_id
        self.grade = grade

    def study(self):
        print(f'{self.name} 正在学习，年级：{self.grade}')

s = Student('李华', 16, '男', '2024001', '高一')
s.speak()
s.study()
print(s.__dict__)

我是李华
李华 正在学习，年级：高一
{'name': '李华', 'age': 16, 'gender': '男', 'stu_id': '2024001', 'grade': '高一'}


In [6]:
# 方法重写（Override）
class Student(Person):
    def __init__(self, name, age, gender, stu_id, grade):
        super().__init__(name, age, gender)
        self.stu_id = stu_id
        self.grade = grade

    def speak(self):
        # 在重写的方法中可以调用父类方法
        super().speak()
        print(f'学号：{self.stu_id}，年级：{self.grade}')

s = Student('张三', 18, '男', '20241001', '高三')
s.speak()

我是张三
学号：20241001，年级：高三


In [7]:
# isinstance 和 issubclass
print(isinstance(s, Student))   # True
print(isinstance(s, Person))    # True
print(issubclass(Student, Person))  # True
print(issubclass(Person, Student))  # False

True
True
True
False


In [8]:
# 多重继承
class Worker:
    def __init__(self, company):
        self.company = company
    def work(self):
        print(f'在 {self.company} 工作')

class PartTimeStudent(Person, Worker):
    def __init__(self, name, age, gender, stu_id, company):
        Person.__init__(self, name, age, gender)
        Worker.__init__(self, company)
        self.stu_id = stu_id

pts = PartTimeStudent('小明', 20, '男', '2024002', '星巴克')
pts.speak()
pts.work()
# 查看方法查找顺序
print(PartTimeStudent.__mro__)

我是小明
在 星巴克 工作
(<class '__main__.PartTimeStudent'>, <class '__main__.Person'>, <class '__main__.Worker'>, <class 'object'>)


## 6. 访问权限控制

Python 通过命名约定来区分访问级别：
- `name`：公有属性
- `_name`：受保护属性（约定，仍可外部访问但不推荐）
- `__name`：私有属性（名称会被改写为 `_ClassName__name`，防止直接访问）

In [9]:
class Person:
    def __init__(self, name, age, idcard):
        self.name = name           # 公有
        self._age = age            # 受保护
        self.__idcard = idcard     # 私有

    def show_info(self):
        print(f'姓名：{self.name}，年龄：{self._age}，身份证：{self.__idcard}')

p = Person('张三', 30, '110101199001011234')
print(p.name)     # 正常访问
print(p._age)     # 可访问，但不推荐
# print(p.__idcard)  # AttributeError
# 实际上私有属性被重命名了
print(p._Person__idcard)   # 强制访问（极其不推荐）
p.show_info()

张三
30
110101199001011234
姓名：张三，年龄：30，身份证：110101199001011234


In [10]:
# getter 和 setter：通过 @property 安全访问属性
class Person:
    def __init__(self, name, age, idcard):
        self.name = name
        self._age = age
        self.__idcard = idcard

    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        if value <= 120:
            self._age = value
        else:
            print('年龄不合法，已设为最大值120')
            self._age = 120

    @property
    def idcard(self):
        # 只显示前6位和后4位，中间隐藏
        return self.__idcard[:6] + '****' + self.__idcard[-4:]

    @idcard.setter
    def idcard(self, value):
        print('身份证号不可修改')

p = Person('李四', 35, '110101199001011234')
print(p.age)      # 35
p.age = 150       # 触发 setter，打印提示并设为120
print(p.age)      # 120
print(p.idcard)   # 110101****1234
p.idcard = 'xxx'  # 禁止修改

35
年龄不合法，已设为最大值120
120
110101****1234
身份证号不可修改


## 7. 魔法方法（Magic Methods）

以 `__xxx__` 命名的特殊方法，由 Python 在特定时机自动调用，如 `__str__`、`__len__`、`__lt__` 等。

In [11]:
class Person:
    def __init__(self, name, age, gender):
        self.name = name
        self.age = age
        self.gender = gender

    def __str__(self):
        return f'{self.name}({self.age}岁, {self.gender})'

    def __len__(self):
        return len(self.__dict__)

    def __lt__(self, other):
        return self.age < other.age

    def __eq__(self, other):
        return self.__dict__ == other.__dict__

    def __getattr__(self, item):
        print(f'你访问的 {item} 属性不存在')

p1 = Person('张三', 18, '男')
p2 = Person('李四', 22, '女')

print(p1)                # 触发 __str__
print(len(p1))           # 触发 __len__
print(p1 < p2)           # 触发 __lt__
print(p1 == p2)          # 触发 __eq__
print(p1.address)        # 触发 __getattr__

张三(18岁, 男)
3
True
False
你访问的 address 属性不存在
None


## 8. object 类 —— 万类之祖

Python 中所有类都隐式继承自 `object`，因此每个对象都拥有 `__str__`、`__class__` 等基本方法。

In [12]:
class MyClass:
    pass

print(issubclass(MyClass, object))  # True
print(isinstance(MyClass(), object))  # True
print(isinstance(100, object))       # True
print(isinstance('hello', object))   # True

# 查看 object 提供的方法
print([m for m in dir(object) if not m.startswith('_')])  # 少量公共方法
print(dir(MyClass))  # 包含 object 的基础方法

True
True
True
True
[]
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__']


## 9. 多态

多态指同一操作作用于不同对象可表现出不同行为。Python 支持两种多态：
- **标准多态**：基于继承，子类重写父类方法。
- **鸭子类型**：“如果它走起来像鸭子，叫起来像鸭子，那它就是鸭子”——不依赖继承，只关心对象是否拥有所需方法。

In [13]:
# 标准多态
class Animal:
    def speak(self):
        print('动物发出声音')

class Dog(Animal):
    def speak(self):
        print('汪汪汪')

class Cat(Animal):
    def speak(self):
        print('喵喵喵')

def make_sound(animal: Animal):
    animal.speak()

make_sound(Dog())   # 汪汪汪
make_sound(Cat())   # 喵喵喵

# 鸭子多态（无需继承）
class Car:
    def speak(self):
        print('嘀嘀嘀')

class Duck:
    def speak(self):
        print('嘎嘎嘎')

def let_speak(obj):
    obj.speak()

let_speak(Car())    # 嘀嘀嘀
let_speak(Duck())   # 嘎嘎嘎

汪汪汪
喵喵喵
嘀嘀嘀
嘎嘎嘎


## 10. 抽象类

抽象类不能直接实例化，用于定义接口规范，强制子类实现特定方法。使用 `abc` 模块的 `ABC` 和 `abstractmethod`。

In [14]:
from abc import ABC, abstractmethod

class MustRun(ABC):
    @abstractmethod
    def run(self):
        pass

class Person(MustRun):
    def __init__(self, name):
        self.name = name
    def run(self):
        print(f'{self.name} 正在奔跑')

# m = MustRun()   # 报错，不能实例化抽象类
p = Person('张三')
p.run()

张三 正在奔跑


## 11. 综合案例：学生成绩管理系统

实现一个命令行交互系统，支持添加学生、删除学生、查看所有学生、录入成绩等功能。运用了类、继承、类属性、实例方法、静态方法等知识。

In [16]:
from datetime import datetime

class Person:
    def __init__(self, name, age, gender):
        self.name = name
        self.age = age
        self.gender = gender

class Student(Person):
    count = 0   # 类属性，用于生成学号

    def __init__(self, name, age, gender):
        super().__init__(name, age, gender)
        Student.count += 1
        self.stu_id = f'{datetime.now().year}{Student.count:03d}'
        self.scores = {}   # 科目:分数

    def add_score(self, subject, score):
        self.scores[subject] = score

    def cal_avg(self):
        if not self.scores:
            return 0.0
        return sum(self.scores.values()) / len(self.scores)

    def __str__(self):
        return f'{self.name}({self.age}-{self.gender}) 成绩：{self.scores} 平均分：{self.cal_avg():.1f}'


class Manager:
    def __init__(self):
        self.stu_list = []

    def add_student(self):
        name = input('姓名：')
        age = int(input('年龄：'))
        gender = input('性别：')
        stu = Student(name, age, gender)
        self.stu_list.append(stu)
        print(f'添加成功，学号：{stu.stu_id}')

    def del_student(self):
        sid = input('请输入要删除的学号：')
        for stu in self.stu_list:
            if stu.stu_id == sid:
                self.stu_list.remove(stu)
                print('删除成功')
                return
        print('未找到该学生')

    def show_all(self):
        if not self.stu_list:
            print('暂无学生')
        else:
            for stu in self.stu_list:
                print(stu)

    def set_score(self):
        sid = input('请输入学号：')
        for stu in self.stu_list:
            if stu.stu_id == sid:
                score_str = input('请输入成绩（格式：科目-分数,科目-分数）：')
                items = score_str.replace('，', ',').split(',')
                for item in items:
                    subject, score = item.split('-')
                    stu.add_score(subject.strip(), float(score.strip()))
                print('成绩录入成功')
                return
        print('未找到该学生')

    def run(self):
        while True:
            print('\n**** 学生管理系统 ****')
            print('1.添加学生  2.删除学生  3.查看所有  4.录入成绩  5.退出')
            choice = input('请选择：')
            if choice == '1':
                self.add_student()
            elif choice == '2':
                self.del_student()
            elif choice == '3':
                self.show_all()
            elif choice == '4':
                self.set_score()
            elif choice == '5':
                print('再见！')
                break
            else:
                print('输入有误，请重新选择')

# 实例化并运行
if __name__ == '__main__':
    mgr = Manager()
    mgr.run()


**** 学生管理系统 ****
1.添加学生  2.删除学生  3.查看所有  4.录入成绩  5.退出
再见！


## 12. 内存分析补充

Python 中变量保存的是对象的引用（内存地址）。不可变对象（int, str, tuple 等）修改时会创建新对象；可变对象（list, dict, 自定义实例）修改时对象地址不变。

In [17]:
# 不可变对象
a = 10
b = a
print(id(a), id(b))   # 相同地址
a = 20
print(id(a), id(b))   # a 地址改变，b 不变

# 可变对象
list1 = [1, 2, 3]
list2 = list1
list1.append(4)
print(id(list1), id(list2))   # 地址相同，内容一起变
print(list1, list2)

140736964610776 140736964610776
140736964611096 140736964610776
2178342772096 2178342772096
[1, 2, 3, 4] [1, 2, 3, 4]


以上便是 Python 面向对象的核心内容。通过本文的学习，你可以系统地掌握类与对象、继承、多态、封装等概念，并能编写出结构清晰、可复用的面向对象程序。